# 4. Ejecutar Pipeline Silver

Propósito: Ejecutar el pipeline silver completo (quality + transform + parquet).

In [2]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: c:\Users\user\Downloads\EP-GDM-G6
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot


In [3]:
from app.utils.spark import SparkClient
spark_client = SparkClient()
spark = spark_client.get_session()

In [4]:
# Metodo C (= "Opcion C" de INSTRUCCIONES_EJECUCION.txt): municipios="drop".
# Lima=C / resto=G, una fila por nombre de municipio (la del menor SEC_EJEC).
# Equivale a:  python main.py silver --drop --municipios drop
MUNICIPIOS = "legacy"

from app.silver import quality
stage = quality.fix_all(spark, municipios=MUNICIPIOS)
print(f"Quality fixes completados (municipios={MUNICIPIOS})")

2026-06-19 15:51:18,322 - INFO - Iniciando correcciones de calidad para todos los datasets
2026-06-19 15:51:18,324 - INFO - Procesando dataset SIAF - Ingreso
2026-06-19 15:51:39,324 - INFO - ingreso_unified: 4337735 filas después de correcciones
2026-06-19 15:52:03,054 - INFO - Guardado: c:\Users\user\Downloads\EP-GDM-G6\data\silver\stage\ingreso_unified.parquet
2026-06-19 15:52:03,058 - INFO - Procesando dataset SISMEPRE
2026-06-19 15:52:03,639 - INFO - rentas_preguntas: 836 filas después de correcciones
2026-06-19 15:52:04,130 - INFO - rentas_formulario: 98 filas después de correcciones
2026-06-19 15:52:05,515 - INFO - rentas_esat_estadistica_atm: 134170 filas después de correcciones
2026-06-19 15:52:06,531 - INFO - rentas_respuestas: 250521 filas después de correcciones
2026-06-19 15:52:06,866 - INFO - rentas_ano_aplicacion: 26 filas después de correcciones
2026-06-19 15:52:06,867 - INFO - Etiquetando categorías municipales (modo=legacy)
2026-06-19 15:52:07,877 - INFO - categorias_m

Quality fixes completados (municipios=legacy)


In [5]:
from app.silver import transforms
dims, facts = transforms.build_all(spark, stage, municipios=MUNICIPIOS)
print(f"Dimensiones: {len(dims)}, Hechos: {len(facts)}")

2026-06-19 15:53:11,463 - INFO - Construyendo dims + hechos para silver parquet (municipios=legacy)
2026-06-19 15:53:11,464 - INFO - Construyendo dimensiones del modelo estrella (municipios=legacy)


2026-06-19 15:53:54,018 - INFO - Dimensiones construidas: 14 dimensiones
2026-06-19 15:53:54,251 - INFO - Construyendo DIM_PREGUNTA_SISMEPRE
2026-06-19 15:53:54,330 - INFO - Construyendo tablas de hechos
2026-06-19 15:54:36,956 - INFO - Tablas de hechos construidas: 3 fact tables


Dimensiones: 15, Hechos: 3


In [6]:
from app.silver.parquet_loader import save_all_to_parquet
from pathlib import Path
save_all_to_parquet(dims, facts, Path("data/silver"))
print("Silver parquet guardados en data/silver/")

2026-06-19 15:55:14,882 - INFO - Saved DIM_TIEMPO to data\silver\DIM_TIEMPO.parquet
2026-06-19 15:55:33,505 - INFO - Saved DIM_EJECUTORA to data\silver\DIM_EJECUTORA.parquet
2026-06-19 15:56:02,272 - INFO - Saved DIM_UBIGEO to data\silver\DIM_UBIGEO.parquet
2026-06-19 15:56:04,757 - INFO - Saved DIM_NIVEL_GOBIERNO to data\silver\DIM_NIVEL_GOBIERNO.parquet
2026-06-19 15:56:05,837 - INFO - Saved DIM_SECTOR to data\silver\DIM_SECTOR.parquet
2026-06-19 15:56:06,778 - INFO - Saved DIM_PLIEGO to data\silver\DIM_PLIEGO.parquet
2026-06-19 15:56:09,724 - INFO - Saved DIM_RUBRO to data\silver\DIM_RUBRO.parquet
2026-06-19 15:56:12,489 - INFO - Saved DIM_TIPO_RECURSO to data\silver\DIM_TIPO_RECURSO.parquet
2026-06-19 15:56:14,043 - INFO - Saved DIM_FUENTE_FINANCIAMIENTO to data\silver\DIM_FUENTE_FINANCIAMIENTO.parquet
2026-06-19 15:56:18,789 - INFO - Saved DIM_GENERICA to data\silver\DIM_GENERICA.parquet
2026-06-19 15:56:22,983 - INFO - Saved DIM_ESPECIFICA to data\silver\DIM_ESPECIFICA.parquet
20

Silver parquet guardados en data/silver/


In [7]:
import os
for f in sorted(os.listdir("data/silver")):
    if f.endswith(".parquet"):
        df = spark.read.parquet(f"data/silver/{f}")
        print(f"{f}: {df.count():,} filas")

DIM_ANIO_APLICACION.parquet: 14 filas
DIM_EJECUTORA.parquet: 1,892 filas
DIM_ESPECIFICA.parquet: 76 filas
DIM_FORMULARIO_SISMEPRE.parquet: 16 filas
DIM_FUENTE_FINANCIAMIENTO.parquet: 4 filas
DIM_GENERICA.parquet: 59 filas
DIM_NIVEL_GOBIERNO.parquet: 1 filas
DIM_PLIEGO.parquet: 1 filas
DIM_PREGUNTA_RENAMU.parquet: 532,538 filas
DIM_PREGUNTA_SISMEPRE.parquet: 236 filas
DIM_RUBRO.parquet: 6 filas
DIM_SECTOR.parquet: 1 filas
DIM_TIEMPO.parquet: 84 filas
DIM_TIPO_RECURSO.parquet: 57 filas
DIM_UBIGEO.parquet: 1,892 filas
FACT_FORMULARIO_SISMEPRE.parquet: 250,521 filas
FACT_INGRESO.parquet: 3,638,741 filas
FACT_RENAMU.parquet: 12,770,291 filas
